# NB2 - Zero-shot LLM routers (Qwen2.5-7B-Instruct, Llama-3.1-8B-Instruct) + their latency (Kaggle T4)

**Settings:** Accelerator = GPU T4, Internet = ON.  **Run:** Run All (or Save Version > Save & Run All).

**Datasets to attach (Add Data):** only one -
1. `KAN_ROUTE_PART4_JOURNAL_READY.zip` (the same dataset you used for NB1)

**Kaggle Secret required:** `HF_TOKEN` (Add-ons > Secrets, then toggle it ON for this notebook). Llama-3.1 is gated: accept the licence on huggingface.co/meta-llama/Llama-3.1-8B-Instruct with the same account first. (If you have the models as Kaggle datasets/models, attach them - the notebook auto-detects folders whose name contains `qwen2.5-7b-instruct` / `llama-3.1-8b-instruct`.)

**What it does:** runs both open-weight models as zero-shot routers (greedy, 4-bit NF4, 28 tool names + one-line definitions in the prompt) on the 372 human queries, the Qwen test set (3,871) and the Llama test set (840); plus (a) query + COCO object-list on the human set, (b) a ranked top-3 prompt on the human set; then measures batch-1 latency on real human queries. Outputs go to `/kaggle/working/KAN_ROUTE_NEUROCOMPUTING_FINAL_EXPERIMENTS/04_zero_shot_llm/` and `/kaggle/working/NB2_OUTPUT.zip`.

**Runtime:** roughly 1-3 h per model on a T4. It is resumable - if the session stops, run again and finished rows are kept (raw responses are saved after every batch). If one model fails (e.g. no Llama access) the other still completes and the failure is written to `llm_run_status.json`.

In [ ]:
import shutil
from pathlib import Path

# Define target directories in the working space
raw_dest = Path(
    "/kaggle/working/KAN_ROUTE_NEUROCOMPUTING_FINAL_EXPERIMENTS/04_zero_shot_llm/raw_responses"
)
d04_dest = Path(
    "/kaggle/working/KAN_ROUTE_NEUROCOMPUTING_FINAL_EXPERIMENTS/04_zero_shot_llm"
)

raw_dest.mkdir(parents=True, exist_ok=True)
d04_dest.mkdir(parents=True, exist_ok=True)

# Copy all checkpoint jsonl files into the working raw_responses folder
copied_count = 0
for p in Path("/kaggle/input").rglob("*.jsonl"):
  shutil.copy(p, raw_dest / p.name)
  print(f"Restored checkpoint: {p.name}")
  copied_count += 1

# Copy any raw latency CSV files into the 04_zero_shot_llm folder
for p in Path("/kaggle/input").rglob("*.csv"):
  if "latency_raw" in p.name or "parsed_predictions" in p.name:
    shutil.copy(p, d04_dest / p.name)
    print(f"Restored file: {p.name}")
    copied_count += 1

print(f"\nSuccessfully restored {copied_count} files to the working directory!")

In [ ]:
# ===== NB2 | Setup: zero-shot LLM routers (Qwen2.5-7B-Instruct, Llama-3.1-8B-Instruct) on a Kaggle T4 =====
import os, sys, re, json, time, math, shutil, random, subprocess, warnings, gc, ast
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
def pip_install(*pk): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pk], check=False)
pip_install("bitsandbytes>=0.43.0", "accelerate")
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
assert torch.cuda.is_available(), "Turn on a GPU: Settings > Accelerator > GPU T4."
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME, "| torch", torch.__version__, "| transformers", transformers.__version__)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
print("HF token found:", HF_TOKEN is not None, "(needed for Llama-3.1; Kaggle > Add-ons > Secrets > HF_TOKEN)")

def find_jr_base():
    for p in Path("/kaggle/input").rglob("PART4_FINAL_REPRODUCIBILITY_AUDIT.md"):
        d = p.parent
        if (d / "part4_predictions_seed_corrected").is_dir() and (d / "qwen_shared28_train.csv").exists(): return d
    raise FileNotFoundError("Attach KAN_ROUTE_PART4_JOURNAL_READY.zip as a Kaggle dataset.")
JR = find_jr_base()
OUT = Path("/kaggle/working/KAN_ROUTE_NEUROCOMPUTING_FINAL_EXPERIMENTS"); D04 = OUT / "04_zero_shot_llm"
RAW = D04 / "raw_responses"; RAW.mkdir(parents=True, exist_ok=True)

# ---- options ----
EVAL_BS = 8                 # batch size for accuracy evaluation (latency is always measured at batch size 1)
RUN_OBJECT_CONTEXT = True   # extra condition on the 372 human queries: query + COCO object list as text
RUN_HUMAN_TOP3 = True       # extra run on the 372 human queries asking for a ranked top-3
LAT_WARMUP, LAT_TIMED = 5, 50
LLMS = {
    "qwen2.5-7b-instruct": dict(hf="Qwen/Qwen2.5-7B-Instruct", hints=["qwen2.5-7b-instruct", "qwen2_5_7b_instruct"], params_b=7.6),
    "llama-3.1-8b-instruct": dict(hf="meta-llama/Llama-3.1-8B-Instruct", hints=["llama-3.1-8b-instruct", "llama_3_1_8b_instruct", "llama-3.1-8b"], params_b=8.0),
}


In [ ]:
# ======================= LIB_EVAL (pure numpy/pandas/sklearn; no torch) =======================
import json, re, math, hashlib
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import f1_score

SEEDS = [42, 123, 2026]
N_CLASSES = 28
ALL_LABELS = list(range(N_CLASSES))

def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def normalize_q(q):
    return re.sub(r"[^\w\s]", "", str(q).lower()).strip()

def macro_f1_fast(y, p, labels):
    """Macro-F1 over `labels` (zero-division -> 0). Equivalent to sklearn f1_score(average='macro', labels=labels)."""
    labels = np.asarray(labels)
    k = int(max(y.max(), p.max(), labels.max())) + 1
    tp = np.bincount(y[y == p], minlength=k).astype(float)
    pc = np.bincount(p, minlength=k).astype(float)
    yc = np.bincount(y, minlength=k).astype(float)
    denom = pc + yc
    f1 = np.divide(2 * tp, denom, out=np.zeros(k), where=denom > 0)
    return float(f1[labels].mean())

# ---------------------------------- prediction files ----------------------------------
def read_pred(path):
    df = pd.read_csv(path)
    df["_top3"] = [json.loads(s) for s in df["pred_top3"]]
    return df

def top3_hit(y, top3_list):
    return np.fromiter((g in t for g, t in zip(y, top3_list)), bool, len(y))

def metrics_from_arrays(y, p, hit3, labels):
    if len(y) == 0:
        return dict(n=0, top1=np.nan, top3=np.nan, macro_f1=np.nan, weighted_f1=np.nan)
    return dict(n=int(len(y)), top1=float((y == p).mean()), top3=float(hit3.mean()),
                macro_f1=macro_f1_fast(y, p, labels),
                weighted_f1=float(f1_score(y, p, average="weighted", labels=labels, zero_division=0)))

# ---------------------------------- human benchmark ----------------------------------
class HumanBench:
    """Loads human_shared28.csv and derives all strata. Fixes the NaN->'nan' bug (acceptable set empty for strict rows)."""
    def __init__(self, human_csv, qtr_csv, ltr_csv, mapping_json):
        h = pd.read_csv(human_csv).reset_index(drop=True)
        self.map = json.load(open(mapping_json))["class_to_local_id"]
        self.inv = {v: k for k, v in self.map.items()}; self.inv[N_CLASSES] = "__INVALID__"
        h["acceptable_tool_labels"] = h["acceptable_tool_labels"].fillna("").astype(str).str.strip()
        strict_true = h["strict_eval"].astype(str).str.lower().isin(["true", "1"]).values
        self.is_amb = strict_true & (h["acceptable_tool_labels"] != "").values
        self.is_strict = strict_true & ~self.is_amb
        assert self.is_strict.sum() + self.is_amb.sum() == len(h)
        self.h = h
        qtr, ltr = pd.read_csv(qtr_csv), pd.read_csv(ltr_csv)
        train_imgs = set(qtr.img_id) | set(ltr.img_id)
        self.known = h["image_id"].isin(train_imgs).values
        assert (self.known == (h["image_pool"] == "known").values).all(), "known/new recomputation disagrees with image_pool"
        self.vis_yes = (h["routing_requires_visual_context"].astype(str).str.lower() == "yes").values
        self.y = h["tool_name"].map(self.map).values.astype(int)
        self.acc_sets = [set(x.strip() for x in s.split(",") if x.strip()) for s in h["acceptable_tool_labels"]]
        self.strata = {"all": np.ones(len(h), bool), "vis_yes": self.vis_yes, "vis_no": ~self.vis_yes,
                       "known": self.known, "new": ~self.known}
        shared = sorted(set(self.y[self.known]) & set(self.y[~self.known]))
        self.shared_labels = shared

    def eval_human(self, df, use_bench_labels=False):
        assert len(df) == len(self.h) and (df["sample_id"].astype(str).values == self.h["query_id"].astype(str).values).all(), \
            "human prediction rows not aligned with query_id"
        y, p, t3 = df["ground_truth"].values.astype(int), df["pred_top1"].values.astype(int), df["_top3"].values
        if use_bench_labels:
            y = self.y.copy()
        else:
            assert (y == self.y).all()
        hit3 = top3_hit(y, t3)
        top1_names = [self.inv[i] for i in p]
        amb1 = np.array([top1_names[i] in self.acc_sets[i] for i in range(len(y))])
        amb3 = np.array([bool({self.inv[c] for c in t3[i]} & self.acc_sets[i]) for i in range(len(y))])
        rows = []
        for name, m in self.strata.items():
            labs = ALL_LABELS if name == "all" else sorted(set(y[m]))
            r = metrics_from_arrays(y[m], p[m], hit3[m], labs)
            r["macro_f1_label_set"] = "28" if name == "all" else "present"
            if name in ("known", "new"):
                r["macro_f1_shared26"] = macro_f1_fast(y[m], p[m], self.shared_labels)
            sm, am = m & self.is_strict, m & self.is_amb
            r.update(strict_n=int(sm.sum()), strict_top1=float((y[sm] == p[sm]).mean()) if sm.any() else np.nan,
                     strict_top3=float(hit3[sm].mean()) if sm.any() else np.nan,
                     ambig_n=int(am.sum()), ambig_top1=float(amb1[am].mean()) if am.any() else np.nan,
                     ambig_top3=float(amb3[am].mean()) if am.any() else np.nan)
            r["cell"] = f"human_{name}"
            rows.append(r)
        return rows

class SyntheticBench:
    """Image-disjoint masks for the two teacher test sets."""
    def __init__(self, qtr_csv, qte_csv, ltr_csv, lte_csv):
        self.qtr, self.qte, self.ltr, self.lte = [pd.read_csv(p) for p in (qtr_csv, qte_csv, ltr_csv, lte_csv)]
        qkeys, lkeys = set(zip(self.qtr.img_id, self.qtr.tool)), set(zip(self.ltr.img_id, self.ltr.tool))
        self.disj = {"qwen_test": np.array([(i, t) not in lkeys for i, t in zip(self.qte.img_id, self.qte.tool)]),
                     "llama_test": np.array([(i, t) not in qkeys for i, t in zip(self.lte.img_id, self.lte.tool)])}

    def eval_synth(self, df, which):
        y, p, t3 = df["ground_truth"].values.astype(int), df["pred_top1"].values.astype(int), df["_top3"].values
        hit3 = top3_hit(y, t3)
        out = []
        for suffix, m in (("", np.ones(len(y), bool)), ("_disjoint", self.disj[which])):
            r = metrics_from_arrays(y[m], p[m], hit3[m], ALL_LABELS)
            r["cell"] = which + suffix
            out.append(r)
        return out

def eval_prediction_set(preds, sbench, hbench):
    """preds: dict with keys qwen_test, llama_test, human -> prediction DataFrames. Returns list of metric rows (cells)."""
    rows = []
    rows += sbench.eval_synth(preds["qwen_test"], "qwen_test")
    rows += sbench.eval_synth(preds["llama_test"], "llama_test")
    rows += hbench.eval_human(preds["human"])
    return rows

def per_class_f1(df):
    y, p = df["ground_truth"].values.astype(int), df["pred_top1"].values.astype(int)
    k = N_CLASSES
    tp = np.bincount(y[y == p], minlength=k).astype(float); pc = np.bincount(p, minlength=k).astype(float); yc = np.bincount(y, minlength=k).astype(float)
    d = pc + yc
    return np.divide(2 * tp, d, out=np.zeros(k), where=d > 0), yc.astype(int)

# ---------------------------------- aggregation ----------------------------------
def sd(x):
    x = np.asarray(x, float)
    return float(np.std(x, ddof=1)) if len(x) > 1 else 0.0

def aggregate_seeds(rows_df, keys, metrics):
    out = []
    for k, g in rows_df.groupby(keys, sort=False):
        rec = dict(zip(keys, k if isinstance(k, tuple) else (k,)))
        for m in metrics:
            v = g.sort_values("seed")[m].values.astype(float)
            rec[f"{m}_mean"], rec[f"{m}_sd"] = float(np.nanmean(v)), sd(v)
            for s, val in zip(sorted(g["seed"].values), v):
                rec[f"{m}_seed{s}"] = float(val)
        rec["n_seeds"] = int(g["seed"].nunique())
        out.append(rec)
    return pd.DataFrame(out)

def paired_delta(rows_df, id_cols, factor, a, b, metrics):
    """Seed-paired delta = metric(factor==a) - metric(factor==b); returns per-seed rows."""
    A = rows_df[rows_df[factor] == a].set_index(id_cols + ["seed"])
    B = rows_df[rows_df[factor] == b].set_index(id_cols + ["seed"])
    idx = A.index.intersection(B.index)
    d = pd.DataFrame(index=idx)
    for m in metrics:
        d[m] = A.loc[idx, m].values - B.loc[idx, m].values
    return d.reset_index()

# ---------------------------------- statistics ----------------------------------
def mcnemar_exact(correct_a, correct_b):
    from scipy.stats import binomtest
    b = int((correct_a & ~correct_b).sum()); c = int((~correct_a & correct_b).sum())
    n = b + c
    p = 1.0 if n == 0 else float(binomtest(min(b, c), n, 0.5, alternative="two-sided").pvalue)
    return b, c, p

def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def paired_bootstrap(preds_a, preds_b, y, B=1000, seed=0, labels=ALL_LABELS):
    """preds_*: list over seeds of int arrays on identical rows. Returns CIs of seed-averaged delta Top-1 and delta macro-F1."""
    rng = np.random.RandomState(seed); n = len(y)
    d1, df1 = [], []
    for _ in range(B):
        idx = rng.randint(0, n, n); yy = y[idx]
        t = np.mean([(pa[idx] == yy).mean() - (pb[idx] == yy).mean() for pa, pb in zip(preds_a, preds_b)])
        f = np.mean([macro_f1_fast(yy, pa[idx], labels) - macro_f1_fast(yy, pb[idx], labels) for pa, pb in zip(preds_a, preds_b)])
        d1.append(t); df1.append(f)
    q = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return q(d1), q(df1)

# ---------------------------------- KAN edge statistics ----------------------------------
def kan_edge_stats(spline_w, in_dim, G=4, lo=-2.0, hi=2.0, npts=200):
    """spline_w: (out_dim, in_dim*G) FastKAN spline_linear weight. Returns per-edge arrays."""
    out_dim = spline_w.shape[0]
    centers = np.linspace(lo, hi, G); rho = (hi - lo) / (G - 1)
    v = np.linspace(lo, hi, npts)
    Bm = np.exp(-(((v[:, None] - centers[None, :]) / rho) ** 2))               # (npts,G)
    W = spline_w.reshape(out_dim, in_dim, G)
    S = np.einsum("pg,oig->oip", Bm, W).reshape(-1, npts)                       # (edges,npts)
    amp = S.max(1) - S.min(1)
    vc = v - v.mean(); sxx = (vc ** 2).sum()
    slope = (S - S.mean(1, keepdims=True)) @ vc / sxx
    fit_ss = (slope ** 2) * sxx
    tot_ss = ((S - S.mean(1, keepdims=True)) ** 2).sum(1)
    r2 = np.divide(fit_ss, tot_ss, out=np.ones_like(fit_ss), where=tot_ss > 1e-12)
    dS = np.diff(S, axis=1)
    mono = (dS >= -1e-9).all(1) | (dS <= 1e-9).all(1)
    return dict(amp=amp.reshape(out_dim, in_dim), r2=r2, mono=mono, out_dim=out_dim, in_dim=in_dim)

def edge_summary(st, topks=(3, 10)):
    r = dict(edges=int(st["r2"].size), near_linear_share=float((st["r2"] >= 0.95).mean()), median_r2=float(np.median(st["r2"])),
             monotone_share=float(st["mono"].mean()), median_amp=float(np.median(st["amp"])))
    A = st["amp"]
    for k in topks:
        if A.shape[1] > k:
            srt = -np.sort(-A, axis=1)
            r[f"top{k}_amp_share"] = float((srt[:, :k].sum(1) / srt.sum(1)).mean())
    return r

# ---------------------------------- LaTeX helpers ----------------------------------
def pm(mu, s, scale=100.0, nd=2):
    return f"{mu*scale:.{nd}f}$\\pm${s*scale:.{nd}f}"

def latex_table(df, caption, label, colfmt=None):
    cols = list(df.columns); colfmt = colfmt or ("l" * len(cols))
    esc = lambda x: str(x).replace("_", "\\_").replace("%", "\\%")
    L = ["\\begin{table}[t]", "\\centering", "\\small", f"\\begin{{tabular}}{{{colfmt}}}", "\\toprule",
         " & ".join(esc(c) for c in cols) + " \\\\", "\\midrule"]
    for _, r in df.iterrows():
        L.append(" & ".join(v if "$\\pm$" in str(v) else esc(v) for v in r.values) + " \\\\")
    L += ["\\bottomrule", "\\end{tabular}", f"\\caption{{{caption}}}", f"\\label{{{label}}}", "\\end{table}"]
    return "\n".join(L)

# ---------------------------------- annotation agreement ----------------------------------
def jaccard(a, b):
    a, b = set(a), set(b)
    return 1.0 if not a and not b else len(a & b) / len(a | b)

def parse_set(s):
    return sorted({x.strip() for x in str(s).replace(";", ",").split(",") if x.strip() and x.strip().lower() != "nan"})

def agreement_report(a, b, tool_col="chosen_tool", amb_col="is_ambiguous", vis_col="routing_requires_visual_context", set_col="acceptable_tool_labels"):
    from sklearn.metrics import cohen_kappa_score
    m = a.merge(b, on="query_id", suffixes=("_1", "_2"))
    res = {"n_items": len(m)}
    res["tool_raw_agreement"] = float((m[f"{tool_col}_1"] == m[f"{tool_col}_2"]).mean())
    res["tool_cohen_kappa"] = float(cohen_kappa_score(m[f"{tool_col}_1"], m[f"{tool_col}_2"]))
    for nm, col in (("ambiguity", amb_col), ("visual_context", vis_col)):
        x, y = m[f"{col}_1"].astype(str).str.lower().str.strip(), m[f"{col}_2"].astype(str).str.lower().str.strip()
        res[f"{nm}_raw_agreement"] = float((x == y).mean())
        res[f"{nm}_cohen_kappa"] = float(cohen_kappa_score(x, y)) if x.nunique() > 1 or y.nunique() > 1 else float("nan")
    s1 = [parse_set(v) for v in m[f"{set_col}_1"]]; s2 = [parse_set(v) for v in m[f"{set_col}_2"]]
    jac = np.array([jaccard(u, v) for u, v in zip(s1, s2)])
    res["acceptable_set_exact_agreement"] = float(np.mean([u == v for u, v in zip(s1, s2)]))
    res["acceptable_set_mean_jaccard"] = float(jac.mean()); res["acceptable_set_median_jaccard"] = float(np.median(jac))
    return res, m


# ---------------------------------- stable-support (21 classes) ----------------------------------
def stable_metrics(y, p, hit3, stable_ids):
    """Only queries whose gold class is stable; F1 averaged over the stable classes; predictions into other classes are errors."""
    m = np.isin(y, stable_ids)
    if not m.any():
        return dict(n=0, top1=np.nan, top3=np.nan, macro_f1=np.nan)
    return dict(n=int(m.sum()), top1=float((y[m] == p[m]).mean()), top3=float(hit3[m].mean()),
                macro_f1=macro_f1_fast(y[m], p[m], np.asarray(stable_ids)))

def add_stable_cells(rows, preds, stable_ids):
    for key, cell in (("qwen_test", "qwen_test_stable21"), ("llama_test", "llama_test_stable21"), ("human", "human_all_stable21")):
        df = preds[key]; y, p = df["ground_truth"].values.astype(int), df["pred_top1"].values.astype(int)
        r = stable_metrics(y, p, top3_hit(y, df["_top3"].values), stable_ids); r["cell"] = cell; rows.append(r)
    return rows

# ---------------------------------- pooled evaluation groups (as in manuscript Table 13) ----------------------------------
GROUPS = {
    "within_teacher": [("qwen", "qwen_test"), ("llama", "llama_test"), ("mixed", "qwen_test"), ("mixed", "llama_test")],
    "cross_teacher": [("qwen", "llama_test"), ("llama", "qwen_test")],
    "image_disjoint": [("qwen", "llama_test_disjoint"), ("mixed", "llama_test_disjoint"), ("llama", "qwen_test_disjoint"), ("mixed", "qwen_test_disjoint")],
    "human_written": [("qwen", "human_all"), ("llama", "human_all"), ("mixed", "human_all")],
    "human_new_image": [("qwen", "human_new"), ("llama", "human_new"), ("mixed", "human_new")],
}

def pooled_group(rows_df, model_col, model, group, metric, groups=None):
    groups = groups or GROUPS
    """Per-seed mean over the group's (regime, cell) pairs -> array over SEEDS."""
    out = []
    for s in SEEDS:
        v = []
        for reg, cell in groups[group]:
            r = rows_df[(rows_df[model_col] == model) & (rows_df["regime"] == reg) & (rows_df["cell"] == cell) & (rows_df["seed"] == s)]
            v.append(float(r[metric].iloc[0]))
        out.append(np.mean(v))
    return np.array(out)

def pooled_compare(rows_df, model_col, a, b, metrics=("top1", "top3", "macro_f1"), groups=None):
    groups = groups or GROUPS
    res = []
    for g in groups:
        for m in metrics:
            va, vb = pooled_group(rows_df, model_col, a, g, m, groups), pooled_group(rows_df, model_col, b, g, m, groups)
            d = va - vb
            res.append(dict(group=g, metric=m, a=a, b=b, a_mean=va.mean(), a_sd=sd(va), b_mean=vb.mean(), b_sd=sd(vb),
                            diff_mean=d.mean(), diff_sd=sd(d), seeds_a_ahead=int((d > 0).sum())))
    return pd.DataFrame(res)

# ---------------------------------- zero-shot LLM output parsing ----------------------------------
REFUSAL_PAT = re.compile(r"(i cannot|i can't|i can not|i'm sorry|i am sorry|unable to|none of the (above|tools)|no (suitable|appropriate) tool|cannot determine)", re.I)

def _norm_tok(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def parse_tool(text, tools):
    """Deterministic parser. Returns (tool_or_None, status) with status in valid_exact/valid_parsed/multiple/refusal/invalid."""
    t = _norm_tok(text)
    if t in tools:
        return t, "valid_exact"
    names = sorted(tools, key=len, reverse=True)
    rest, found = f"_{t}_", []
    for n in names:                                     # longest first so a short name inside a longer one is not double-counted
        if f"_{n}_" in rest:
            found.append(n); rest = rest.replace(f"_{n}_", "__")
    if len(found) == 1:
        return found[0], "valid_parsed"
    if len(found) > 1:
        return None, "multiple"
    return None, ("refusal" if REFUSAL_PAT.search(str(text)) else "invalid")

def parse_top3(text, tools):
    """Ordered list of up to 3 distinct valid tool names in order of appearance."""
    found = []
    for part in re.split(r"[,\n;]| and ", str(text)):
        tool, st = parse_tool(part, tools)
        if tool is not None and tool not in found:
            found.append(tool)
    if not found:                                       # fall back to order-of-appearance scan of the whole text
        t = f"_{_norm_tok(text)}_"
        pos = sorted((t.find(f"_{n}_"), n) for n in tools if f"_{n}_" in t)
        found = [n for _, n in pos]
    return found[:3]


GROUPS_STABLE = {
    "within_teacher": [(r, c + "_stable21") for r, c in GROUPS["within_teacher"]],
    "cross_teacher": [(r, c + "_stable21") for r, c in GROUPS["cross_teacher"]],
    "human_written": [(r, "human_all_stable21") for r in ("qwen", "llama", "mixed")],
}

def latex_table_resized(df, caption, label):
    t = latex_table(df, caption, label)
    return t.replace("\\begin{tabular}", "\\resizebox{\\textwidth}{!}{%\n\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}}", 1)


# Written once, BEFORE any zero-shot LLM or independent-annotation results were seen. Shared by NB2 (LLM prompt) and NB3 (annotation guide).
TOOL_DEFS = {
 "3d_reconstruction": "Build a 3D model, mesh or point cloud of the scene or of an object from the image.",
 "action_recognition": "Identify what action or activity people or animals are performing (running, cooking, riding).",
 "age_gender_estimation": "Estimate the age and/or gender of the people in the image.",
 "animal_detection": "Find and locate animals in the image (which animals and where).",
 "anomaly_detection": "Find unusual, abnormal, defective or out-of-place objects or regions in the image.",
 "attribute_recognition": "Recognize attributes of an object or person such as colour, material, clothing, texture or state.",
 "background_removal": "Remove or replace the background and keep the foreground subject as a cut-out.",
 "depth_estimation": "Estimate depth or distance from the camera (how far things are, which is closer).",
 "face_detection": "Detect and locate human faces in the image.",
 "facial_expression_recognition": "Recognize the emotion or expression on a person's face (smiling, angry, surprised).",
 "fine_grained_classification": "Identify the specific subtype of an object (dog breed, bird species, car model, plant variety).",
 "image_captioning": "Write a general natural-language description (caption) of the whole image.",
 "image_classification": "Assign the whole image a single overall category or label (what kind of image / main object class).",
 "instance_segmentation": "Segment each individual object separately with its own pixel mask.",
 "logo_detection": "Detect and recognize brand logos in the image.",
 "object_counting": "Count how many objects of a given kind are in the image.",
 "object_detection": "Detect and locate objects of various categories with bounding boxes.",
 "person_detection": "Detect and locate people (pedestrians) in the image.",
 "pose_estimation": "Estimate the body pose of people (skeleton / joint keypoints, posture).",
 "relation_detection": "Identify relationships between objects or people (person riding horse, cup on table).",
 "salient_object_segmentation": "Segment the most visually prominent object(s) in the image.",
 "scene_recognition": "Recognize the type of scene or place (kitchen, beach, street, indoor or outdoor).",
 "semantic_segmentation": "Label every pixel with a semantic class (road, sky, building) without separating instances.",
 "style_transfer": "Re-render the image in the style of an artwork, an artist or another visual style.",
 "super_resolution": "Increase resolution, sharpen, denoise or enhance a low-quality image.",
 "vehicle_detection": "Detect and locate vehicles (cars, trucks, buses, motorbikes, bicycles).",
 "visual_question_answering": "Answer a specific factual question about the content of the image.",
 "visual_reasoning": "Answer questions that need reasoning over the image: spatial or logical relations, comparison, inference about what is happening or why.",
}
BOUNDARY_NOTES = [
 "image_captioning vs visual_question_answering: captioning describes the whole image; VQA answers one specific question.",
 "image_classification vs object_detection vs fine_grained_classification: classification labels the whole image; detection locates objects; fine-grained names the specific subtype.",
 "semantic_segmentation vs instance_segmentation vs salient_object_segmentation: semantic labels every pixel by class; instance separates each object; salient picks the most prominent object.",
 "person_detection vs face_detection vs pose_estimation: whole people; faces only; body joints/posture.",
 "visual_question_answering vs visual_reasoning: a direct factual lookup vs a question needing spatial/logical inference.",
 "scene_recognition vs image_classification vs image_captioning: type of place; overall category; free-text description.",
 "depth_estimation vs 3d_reconstruction: distance/ordering of things in view vs building a 3D model.",
]
assert len(TOOL_DEFS) == 28


In [ ]:
# ===== NB2 | Evaluation sets, prompts, object lists =====
import json as _json
hb = HumanBench(JR / "human_shared28.csv", JR / "qwen_shared28_train.csv", JR / "llama_shared28_train.csv", JR / "shared_28_class_mapping.json")
sb = SyntheticBench(JR / "qwen_shared28_train.csv", JR / "qwen_shared28_test.csv", JR / "llama_shared28_train.csv", JR / "llama_shared28_test.csv")
TOOL2ID = hb.map; ID2TOOL = {v: k for k, v in TOOL2ID.items()}; TOOLS = set(TOOL2ID)
STABLE = [TOOL2ID[c] for c in (JR / "stable_support_classes.txt").read_text().split()]
assert set(TOOL_DEFS) == TOOLS
def parse_objs(v):
    if isinstance(v, str) and v.strip().startswith("["):
        try: return list(ast.literal_eval(v))
        except Exception: return []
    if isinstance(v, str): return [x.strip() for x in v.split(",") if x.strip()]
    return []
qte, lte = sb.qte, sb.lte
EVALSETS = {
    "human": pd.DataFrame(dict(sample_id=hb.h.query_id.astype(str).values, query=hb.h["query"].values, y=hb.y, objects=[parse_objs(v) for v in hb.h["objects"]])),
    "qwen_test": pd.DataFrame(dict(sample_id=[str(i) for i in range(len(qte))], query=qte["query"].values, y=qte.tool.map(TOOL2ID).values, objects=[[]] * len(qte))),
    "llama_test": pd.DataFrame(dict(sample_id=[str(i) for i in range(len(lte))], query=lte["query"].values, y=lte.tool.map(TOOL2ID).values, objects=[[]] * len(lte))),
}
print({k: len(v) for k, v in EVALSETS.items()}, "| human rows with an object list:", int(sum(len(o) > 0 for o in EVALSETS["human"].objects)))

TOOL_LIST_TXT = "\n".join(f"- {k}: {v}" for k, v in TOOL_DEFS.items())
SYS_PROMPT = ("You are the routing module of a computer-vision agent. Given a user's request about an image, choose the most appropriate tool "
              "from the list. Use only tool names from the list.")
def build_messages(query, objects=None, top3=False):
    ctx = f"Objects visible in the image: {', '.join(objects)}\n\n" if objects else ""
    instr = ("Reply with the three most appropriate tool names, most likely first, separated by commas, and nothing else." if top3
             else "Reply with exactly one tool name from the list and nothing else.")
    user = f"Available tools:\n{TOOL_LIST_TXT}\n\n{ctx}User request: {query}\n\n{instr}"
    return [{"role": "system", "content": SYS_PROMPT}, {"role": "user", "content": user}]
json.dump(TOOL_DEFS, open(D04 / "tool_definitions.json", "w"), indent=2)
json.dump(dict(system=SYS_PROMPT, example_query_only=build_messages("<QUERY>"), example_object_context=build_messages("<QUERY>", ["<obj1>", "<obj2>"]),
               example_top3=build_messages("<QUERY>", None, True), decoding="greedy, max_new_tokens 24 (top-1) / 48 (top-3)", boundary_notes_NOT_in_prompt=BOUNDARY_NOTES),
          open(D04 / "prompt_templates.json", "w"), indent=2)


In [ ]:
# ===== NB2 | LLM loading, batched generation (resumable), latency =====
def resolve_model(spec):
    for p in Path("/kaggle/input").rglob("config.json"):
        low = str(p.parent).lower()
        if any(h in low for h in spec["hints"]) and any(p.parent.glob("*.safetensors")): return str(p.parent)
    return spec["hf"]

def load_llm(name):
    spec = LLMS[name]; path = resolve_model(spec); print(f"loading {name} from {path} (4-bit NF4, fp16 compute)")
    tok = AutoTokenizer.from_pretrained(path, token=HF_TOKEN); tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(path, quantization_config=bnb, device_map={"": 0}, token=HF_TOKEN, torch_dtype=torch.float16)
    model.eval(); gc_ = model.generation_config
    gc_.do_sample = False; gc_.temperature = None; gc_.top_p = None; gc_.top_k = None
    return tok, model, path

def render(tok, msgs): return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)

@torch.inference_mode()
def generate_batch(tok, model, prompts, max_new):
    try:
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(prompts) == 1: raise
        h = len(prompts) // 2
        return generate_batch(tok, model, prompts[:h], max_new) + generate_batch(tok, model, prompts[h:], max_new)

def run_eval(name, tok, model, cond, evalset, top3=False):
    items = EVALSETS[evalset]; path = RAW / f"{name}__{cond}__{evalset}{'__top3' if top3 else ''}.jsonl"
    done = {}
    if path.exists():
        for ln in open(path):
            r = json.loads(ln); done[r["sample_id"]] = r["response"]
    use_obj = (cond == "objects")
    prompts = {sid: render(tok, build_messages(q, o if use_obj else None, top3)) for sid, q, o in zip(items.sample_id, items["query"], items.objects)}
    todo = sorted([s for s in items.sample_id if s not in done], key=lambda s: len(prompts[s]))
    t0 = time.time()
    with open(path, "a") as f:
        for i in range(0, len(todo), EVAL_BS):
            ids = todo[i:i + EVAL_BS]
            outs = generate_batch(tok, model, [prompts[s] for s in ids], 48 if top3 else 24)
            for s, o in zip(ids, outs): f.write(json.dumps(dict(sample_id=s, response=o)) + "\n")
            f.flush()
            if (i // EVAL_BS) % 25 == 0: print(f"  {name} {cond} {evalset}{' top3' if top3 else ''}: {min(i+EVAL_BS, len(todo))}/{len(todo)} new rows, {(time.time()-t0)/60:.1f} min", flush=True)
    return path

@torch.inference_mode()
def measure_latency(name, tok, model):
    sub = EVALSETS["human"].sample(n=LAT_WARMUP + LAT_TIMED, random_state=0)
    rows = []
    for k, (q, o) in enumerate(zip(sub["query"], sub.objects)):
        torch.cuda.synchronize(); t0 = time.perf_counter()
        prompt = render(tok, build_messages(q, None, False))
        enc = tok([prompt], return_tensors="pt", add_special_tokens=False).to(model.device)
        out = model.generate(**enc, max_new_tokens=24, do_sample=False, pad_token_id=tok.pad_token_id)
        text = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        torch.cuda.synchronize(); ms = (time.perf_counter() - t0) * 1000
        if k >= LAT_WARMUP: rows.append(dict(model=name, i=k - LAT_WARMUP, ms=ms, prompt_tokens=int(enc["input_ids"].shape[1]), new_tokens=int(out.shape[1] - enc["input_ids"].shape[1])))
    return pd.DataFrame(rows)


In [ ]:
# ===== NB2 | Run both LLMs (each model: accuracy sets -> object-context -> top-3 -> latency). Resumable. =====
RUN_STATUS, LAT_ALL = {}, []
for name in LLMS:
    try:
        tok, model, path = load_llm(name)
        for evalset in ("human", "qwen_test", "llama_test"): run_eval(name, tok, model, "query_only", evalset)
        if RUN_OBJECT_CONTEXT: run_eval(name, tok, model, "objects", "human")
        if RUN_HUMAN_TOP3: run_eval(name, tok, model, "query_only", "human", top3=True)
        lat = measure_latency(name, tok, model); lat.to_csv(D04 / f"llm_router_latency_raw_{name}.csv", index=False); LAT_ALL.append(lat)
        RUN_STATUS[name] = dict(ok=True, model_path=path, quantization="bitsandbytes NF4 4-bit, fp16 compute")
    except Exception as e:
        RUN_STATUS[name] = dict(ok=False, error=repr(e)[:500]); print("FAILED:", name, repr(e)[:300])
    finally:
        for v in ("model", "tok"):
            if v in globals(): del globals()[v]
        gc.collect(); torch.cuda.empty_cache()
json.dump(RUN_STATUS, open(D04 / "llm_run_status.json", "w"), indent=2); print(RUN_STATUS)


In [ ]:
# ===== NB2 | Parse -> metrics -> tables (pure CPU; re-runnable at any time on the saved raw responses) =====
def load_raw(name, cond, evalset, top3=False):
    p = RAW / f"{name}__{cond}__{evalset}{'__top3' if top3 else ''}.jsonl"
    if not p.exists(): return None
    d = {}
    for ln in open(p):
        r = json.loads(ln); d[r["sample_id"]] = r["response"]
    return d

def to_pred_df(items, raw, top3=False):
    codes, tops, status, parsed = [], [], [], []
    for sid in items.sample_id:
        resp = raw.get(sid, "")
        if top3:
            lst = parse_top3(resp, TOOLS); ids = [TOOL2ID[t] for t in lst] + [N_CLASSES] * (3 - len(lst))
            codes.append(ids[0]); tops.append(ids); status.append("valid" if lst else "invalid"); parsed.append(",".join(lst))
        else:
            tool, st = parse_tool(resp, TOOLS); c = TOOL2ID[tool] if tool else N_CLASSES
            codes.append(c); tops.append([c, N_CLASSES, N_CLASSES]); status.append(st); parsed.append(tool or "")
    df = pd.DataFrame(dict(sample_id=items.sample_id.values, ground_truth=items.y.values.astype(int), pred_top1=codes))
    df["_top3"] = tops
    return df, pd.DataFrame(dict(sample_id=items.sample_id.values, query=items["query"].values, gold_tool=[ID2TOOL[int(y)] for y in items.y],
                                 raw_response=[raw.get(s, "") for s in items.sample_id], parsed=parsed, status=status))

RESULTS, TOP3_RESULTS, STATUS_ROWS = [], [], []
for name in LLMS:
    if not RUN_STATUS.get(name, {}).get("ok", False) and not any(RAW.glob(f"{name}__*.jsonl")): continue
    preds = {}
    for evalset in ("human", "qwen_test", "llama_test"):
        raw = load_raw(name, "query_only", evalset)
        if raw is None or len(raw) < len(EVALSETS[evalset]): print("incomplete:", name, evalset, None if raw is None else len(raw)); continue
        df, det = to_pred_df(EVALSETS[evalset], raw); preds[evalset] = df
        det.to_csv(D04 / f"parsed_predictions_{name}_query_only_{evalset}.csv", index=False)
        sc = det.status.value_counts().to_dict(); n = len(det)
        STATUS_ROWS.append(dict(model=name, condition="query_only", evalset=evalset, n=n, valid_rate=(sc.get("valid_exact", 0) + sc.get("valid_parsed", 0)) / n,
                                **{k: sc.get(k, 0) for k in ("valid_exact", "valid_parsed", "multiple", "refusal", "invalid")}))
    rows = []
    if "qwen_test" in preds: rows += sb.eval_synth(preds["qwen_test"], "qwen_test")
    if "llama_test" in preds: rows += sb.eval_synth(preds["llama_test"], "llama_test")
    if "human" in preds: rows += hb.eval_human(preds["human"])
    if len(preds) == 3: rows = add_stable_cells(rows, preds, STABLE)
    for r in rows:
        for k in ("top3", "strict_top3", "ambig_top3"): r[k] = np.nan          # a single-label reply has no ranking
        r.update(model=name, condition="query_only")
    RESULTS += rows
    raw = load_raw(name, "objects", "human")
    if raw is not None and len(raw) == len(EVALSETS["human"]):
        df, det = to_pred_df(EVALSETS["human"], raw); det.to_csv(D04 / f"parsed_predictions_{name}_objects_human.csv", index=False)
        sc = det.status.value_counts().to_dict(); n = len(det)
        STATUS_ROWS.append(dict(model=name, condition="objects", evalset="human", n=n, valid_rate=(sc.get("valid_exact", 0) + sc.get("valid_parsed", 0)) / n, **{k: sc.get(k, 0) for k in ("valid_exact", "valid_parsed", "multiple", "refusal", "invalid")}))
        rr = hb.eval_human(df)
        for r in rr:
            for k in ("top3", "strict_top3", "ambig_top3"): r[k] = np.nan
            r.update(model=name, condition="objects")
        RESULTS += rr
    raw = load_raw(name, "query_only", "human", top3=True)
    if raw is not None and len(raw) == len(EVALSETS["human"]):
        df, det = to_pred_df(EVALSETS["human"], raw, top3=True); det.to_csv(D04 / f"parsed_predictions_{name}_top3_human.csv", index=False)
        for r in hb.eval_human(df): r.update(model=name, condition="top3_prompt"); TOP3_RESULTS.append(r)

LLM_RES = pd.DataFrame(RESULTS); STAT = pd.DataFrame(STATUS_ROWS)
keep = ["model", "condition", "cell", "n", "top1", "macro_f1", "weighted_f1", "strict_n", "strict_top1", "ambig_n", "ambig_top1", "macro_f1_shared26"]
LLM_RES = LLM_RES.merge(STAT[STAT.condition == "query_only"].groupby("model").valid_rate.mean().rename("mean_valid_rate"), on="model", how="left") if len(STAT) else LLM_RES
LLM_RES[[c for c in keep + ["mean_valid_rate"] if c in LLM_RES.columns]].to_csv(D04 / "llm_router_results.csv", index=False)
STAT.to_csv(D04 / "llm_router_output_validity.csv", index=False)
pd.DataFrame(TOP3_RESULTS).to_csv(D04 / "llm_router_human_top3.csv", index=False)

lat_files = sorted(D04.glob("llm_router_latency_raw_*.csv"))
if lat_files:
    L = pd.concat([pd.read_csv(f) for f in lat_files]); env = dict(gpu=GPU_NAME, torch=torch.__version__, transformers=transformers.__version__, batch_size=1, new_tokens_cap=24, quantization="NF4 4-bit")
    LS = L.groupby("model").agg(n=("ms", "size"), mean_ms=("ms", "mean"), sd_ms=("ms", lambda x: sd(x)), median_ms=("ms", "median"), p95_ms=("ms", lambda x: np.percentile(x, 95)),
                                mean_prompt_tokens=("prompt_tokens", "mean"), mean_new_tokens=("new_tokens", "mean")).reset_index()
    LS.to_csv(D04 / "llm_router_latency.csv", index=False); json.dump(env, open(D04 / "llm_router_latency_environment.json", "w"), indent=2)

def f(model, cell, col):
    r = LLM_RES[(LLM_RES.model == model) & (LLM_RES.condition == "query_only") & (LLM_RES.cell == cell)]
    return float(r[col].iloc[0]) if len(r) else float("nan")
md = ["# Zero-Shot LLM Router Baseline (computed)\n",
      "Protocol: greedy decoding; 28 tool names + one-line definitions in the prompt (identical text to the annotation guide, written before any result was seen); open-weight models run locally "
      "in 4-bit NF4 on one T4; no fine-tuning, no few-shot examples. Invalid, multi-tool and refusal outputs count as errors. Single-label replies have no ranking, so Top-3 is only reported from the separate top-3 prompt run on the human set.\n",
      "**Circularity caveat:** Qwen2.5-7B generated the Qwen corpus and Llama-3.1-8B the Llama corpus; zero-shot scores on a model's own corpus are a teacher-reading-its-own-labels measurement, not an independent one.\n",
      "| Model | Qwen test Top-1 | Llama test Top-1 | Human all Top-1 | Human strict Top-1 | Human ambig-aware Top-1 | Human macro-F1 | valid-output rate (query only) |", "|---|---|---|---|---|---|---|---|"]
for name in LLMS:
    if LLM_RES.empty or name not in set(LLM_RES.model): continue
    vr = STAT[(STAT.model == name) & (STAT.condition == "query_only")].valid_rate.mean()
    md.append(f"| {name} | {f(name,'qwen_test','top1')*100:.2f} | {f(name,'llama_test','top1')*100:.2f} | {f(name,'human_all','top1')*100:.2f} | {f(name,'human_all','strict_top1')*100:.2f} | {f(name,'human_all','ambig_top1')*100:.2f} | {f(name,'human_all','macro_f1')*100:.2f} | {vr*100:.1f}% |")
if lat_files:
    md.append("\n## Latency (batch 1, real human queries, T4; prompt build + tokenise + generate + decode)\n"); md.append(LS.round(1).to_markdown(index=False))
    md.append("\nThese are per-request LLM generation latencies in 4-bit on the T4. They are not router-only latencies; compare with the trained heads only through NB3, which measures both on the same hardware signature.")
(D04 / "ZERO_SHOT_LLM_BASELINE_SUMMARY.md").write_text("\n".join(md)); print("\n".join(md))
rows = [{"Model": n, "Qwen test": pm(f(n, "qwen_test", "top1"), 0), "Llama test": pm(f(n, "llama_test", "top1"), 0), "Human all": pm(f(n, "human_all", "top1"), 0), "Human new-image": pm(f(n, "human_new", "top1"), 0)} for n in LLMS if not LLM_RES.empty and n in set(LLM_RES.model)]
if rows:
    t = pd.DataFrame(rows).replace(r"\$\\pm\$0\.00", "", regex=True)
    (D04 / "table_llm_baseline.tex").write_text(latex_table(t, "Zero-shot LLM router Top-1 (\\%); single run per model (deterministic greedy decoding, no seeds).", "tab:llm_baseline"))


In [ ]:
(OUT / "README_NB2.md").write_text(f"NB2 outputs: 04_zero_shot_llm. GPU: {GPU_NAME}. Models run in 4-bit NF4. Run status: {json.dumps(RUN_STATUS)}\n")
shutil.make_archive("/kaggle/working/NB2_OUTPUT", "zip", OUT)
print("DONE. Download /kaggle/working/NB2_OUTPUT.zip (or Save Version and attach this notebook's output to NB3).")
